In [1]:

import sqlite3, os, warnings
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import ipywidgets as widgets
from IPython.display import display, clear_output, HTML
from datetime import date, datetime
import numpy as np

warnings.filterwarnings('ignore')
print("✅ Libraries loaded successfully.")

✅ Libraries loaded successfully.


In [2]:

# ─────────────────────────────────────────────
DB_PATH = "student_erp.db"
CURRENT_YEAR  = datetime.now().year
DEPARTMENTS   = ["Computer Science", "Electronics", "Mechanical", "Civil", "Business"]
SEMESTERS     = [f"Sem {i}" for i in range(1, 9)]
GRADE_SCALE   = {"O":10,"A+":9,"A":8,"B+":7,"B":6,"C":5,"F":0}

THEME = {
    "primary":   "#4A90D9",
    "success":   "#27AE60",
    "danger":    "#E74C3C",
    "warning":   "#F39C12",
    "dark":      "#2C3E50",
    "light_bg":  "#ECF0F1",
}

SUBJECTS_MAP = {
    "Computer Science": ["Data Structures","Algorithms","DBMS","OS","Networks","AI"],
    "Electronics":      ["Circuits","Signals","Microprocessors","VLSI","Communication","Control Systems"],
    "Mechanical":       ["Thermodynamics","Fluid Mechanics","Manufacturing","CAD","Dynamics","Materials"],
    "Civil":            ["Structures","Soil Mech","Survey","Hydraulics","Concrete","Transportation"],
    "Business":         ["Accounting","Marketing","Finance","HRM","Strategy","Economics"],
}

def get_conn(): return sqlite3.connect(DB_PATH)
print("✅ Configuration ready.")

✅ Configuration ready.


In [3]:

def create_tables():
    conn = get_conn()
    c = conn.cursor()

    c.execute("""CREATE TABLE IF NOT EXISTS students (
        student_id   TEXT PRIMARY KEY,
        name         TEXT NOT NULL,
        dob          TEXT,
        gender       TEXT,
        department   TEXT,
        semester     TEXT,
        email        TEXT UNIQUE,
        phone        TEXT,
        address      TEXT,
        enrolled_on  TEXT DEFAULT CURRENT_DATE,
        status       TEXT DEFAULT 'Active'
    )""")

    c.execute("""CREATE TABLE IF NOT EXISTS attendance (
        id           INTEGER PRIMARY KEY AUTOINCREMENT,
        student_id   TEXT,
        subject      TEXT,
        date         TEXT,
        status       TEXT,
        FOREIGN KEY(student_id) REFERENCES students(student_id)
    )""")

    c.execute("""CREATE TABLE IF NOT EXISTS marks (
        id           INTEGER PRIMARY KEY AUTOINCREMENT,
        student_id   TEXT,
        subject      TEXT,
        semester     TEXT,
        internal     REAL DEFAULT 0,
        midterm      REAL DEFAULT 0,
        external     REAL DEFAULT 0,
        total        REAL GENERATED ALWAYS AS
                        (ROUND(internal*0.2 + midterm*0.3 + external*0.5, 2)) STORED,
        grade        TEXT,
        FOREIGN KEY(student_id) REFERENCES students(student_id)
    )""")

    c.execute("""CREATE TABLE IF NOT EXISTS courses (
        id           INTEGER PRIMARY KEY AUTOINCREMENT,
        course_code  TEXT UNIQUE,
        course_name  TEXT,
        department   TEXT,
        credits      INTEGER DEFAULT 3,
        semester     TEXT
    )""")

    c.execute("""CREATE TABLE IF NOT EXISTS admin_log (
        id           INTEGER PRIMARY KEY AUTOINCREMENT,
        action       TEXT,
        performed_by TEXT DEFAULT 'Admin',
        timestamp    TEXT DEFAULT CURRENT_TIMESTAMP
    )""")

    conn.commit(); conn.close()
    print("✅ All tables created successfully.")

create_tables()

✅ All tables created successfully.


In [4]:

def seed_demo_data():
    conn = get_conn()
    c = conn.cursor()

    students = [
        ("STU001","Arjun Sharma","2002-05-14","Male","Computer Science","Sem 3","arjun@uni.edu","9876543210","Delhi"),
        ("STU002","Priya Patel","2003-01-22","Female","Electronics","Sem 2","priya@uni.edu","9876543211","Mumbai"),
        ("STU003","Rahul Verma","2001-11-30","Male","Mechanical","Sem 5","rahul@uni.edu","9876543212","Chennai"),
        ("STU004","Sneha Gupta","2002-08-17","Female","Computer Science","Sem 3","sneha@uni.edu","9876543213","Bangalore"),
        ("STU005","Amit Singh","2003-03-09","Male","Business","Sem 1","amit@uni.edu","9876543214","Kolkata"),
        ("STU006","Deepika Rao","2001-07-25","Female","Civil","Sem 6","deepika@uni.edu","9876543215","Hyderabad"),
    ]
    c.executemany("""INSERT OR IGNORE INTO students
        (student_id,name,dob,gender,department,semester,email,phone,address)
        VALUES (?,?,?,?,?,?,?,?,?)""", students)

    import random, calendar
    random.seed(42)
    for sid, _, _, _, dept, sem, *_ in students:
        subjects = SUBJECTS_MAP[dept]
        for subj in subjects:
            for day in range(1, 16):
                try:
                    d = f"2024-01-{day:02d}"
                    st = random.choice(["Present","Present","Present","Absent"])
                    c.execute("INSERT OR IGNORE INTO attendance (student_id,subject,date,status) VALUES (?,?,?,?)",
                              (sid, subj, d, st))
                except: pass
            internal = round(random.uniform(14,20),1)
            midterm  = round(random.uniform(20,30),1)
            external = round(random.uniform(35,50),1)
            total    = round(internal*0.2 + midterm*0.3 + external*0.5, 2)
            if   total >= 90: grade = "O"
            elif total >= 80: grade = "A+"
            elif total >= 70: grade = "A"
            elif total >= 60: grade = "B+"
            elif total >= 50: grade = "B"
            elif total >= 40: grade = "C"
            else:             grade = "F"
            c.execute("""INSERT OR IGNORE INTO marks
                (student_id,subject,semester,internal,midterm,external,grade)
                VALUES (?,?,?,?,?,?,?)""", (sid,subj,sem,internal,midterm,external,grade))

    conn.commit(); conn.close()
    print("✅ Demo data seeded: 6 students | Attendance | Marks")

seed_demo_data()

✅ Demo data seeded: 6 students | Attendance | Marks


In [5]:

def query_df(sql, params=()):
    conn = get_conn()
    df = pd.read_sql_query(sql, conn, params=params)
    conn.close()
    return df

def execute_query(sql, params=()):
    conn = get_conn(); c = conn.cursor()
    c.execute(sql, params)
    conn.commit(); conn.close()

def log_action(action):
    execute_query("INSERT INTO admin_log (action) VALUES (?)", (action,))

def get_student_ids():
    df = query_df("SELECT student_id || ' - ' || name as label, student_id FROM students ORDER BY name")
    return list(df['label']), list(df['student_id'])

def styled_html(title, color=THEME['primary']):
    return HTML(f"""<div style="background:{color};color:white;padding:10px 18px;
        border-radius:8px;font-size:16px;font-weight:bold;margin:8px 0">{title}</div>""")

print("✅ Helper utilities ready.")

✅ Helper utilities ready.


In [6]:

def make_text(desc, val="", ph=""):
    return widgets.Text(description=desc, value=val, placeholder=ph,
                        layout=widgets.Layout(width="380px"),
                        style={"description_width":"130px"})

def make_dropdown(desc, opts, val=None):
    return widgets.Dropdown(description=desc, options=opts,
                            value=val or opts[0],
                            layout=widgets.Layout(width="380px"),
                            style={"description_width":"130px"})

def make_btn(desc, color="#4A90D9", icon=""):
    return widgets.Button(description=desc, button_style="",
                          icon=icon,
                          layout=widgets.Layout(width="180px", height="36px"),
                          style={"button_color": color, "font_weight":"bold"})

def make_output():
    return widgets.Output(layout=widgets.Layout(
        border="1px solid #ddd", border_radius="8px",
        padding="10px", min_height="60px", max_height="350px",
        overflow_y="auto"))

def divider():
    display(widgets.HTML('<hr style="border:1px solid #ccc;margin:8px 0">'))

print("✅ Widget factory ready.")

✅ Widget factory ready.


In [7]:

display(styled_html("👤 STUDENT REGISTRATION", THEME['primary']))

reg_id   = make_text("Student ID",  ph="e.g. STU007")
reg_name = make_text("Full Name",   ph="e.g. Rajesh Kumar")
reg_dob  = make_text("Date of Birth", ph="YYYY-MM-DD")
reg_mail = make_text("Email",       ph="student@uni.edu")
reg_phone= make_text("Phone",       ph="10-digit number")
reg_addr = make_text("Address",     ph="City, State")

reg_gender = make_dropdown("Gender",     ["Male","Female","Other"])
reg_dept   = make_dropdown("Department", DEPARTMENTS)
reg_sem    = make_dropdown("Semester",   SEMESTERS)

reg_out  = make_output()

left_col  = widgets.VBox([reg_id, reg_name, reg_dob, reg_mail])
right_col = widgets.VBox([reg_gender, reg_dept, reg_sem, reg_phone])

display(widgets.HBox([left_col, right_col]))
display(reg_addr)


Text(value='', description='Address', layout=Layout(width='380px'), placeholder='City, State', style=TextStyle…

In [8]:

btn_reg   = make_btn("➕ Register", THEME['success'])
btn_clear = make_btn("🔄 Clear",    THEME['warning'])

def on_register(b):
    with reg_out:
        clear_output()
        fields = [reg_id.value, reg_name.value, reg_dob.value, reg_mail.value]
        if any(f.strip() == "" for f in fields):
            display(HTML('<p style="color:red;font-weight:bold">❌ ID, Name, DOB, Email are required!</p>'))
            return
        try:
            execute_query("""INSERT INTO students
                (student_id,name,dob,gender,department,semester,email,phone,address)
                VALUES (?,?,?,?,?,?,?,?,?)""",
                (reg_id.value.strip(), reg_name.value.strip(), reg_dob.value.strip(),
                 reg_gender.value, reg_dept.value, reg_sem.value,
                 reg_mail.value.strip(), reg_phone.value.strip(), reg_addr.value.strip()))
            log_action(f"Registered student: {reg_id.value}")
            display(HTML(f'<p style="color:green;font-weight:bold">✅ Student {reg_name.value} registered successfully!</p>'))
        except sqlite3.IntegrityError as e:
            display(HTML(f'<p style="color:red">❌ Error: {e}</p>'))

def on_clear(b):
    for w in [reg_id,reg_name,reg_dob,reg_mail,reg_phone,reg_addr]:
        w.value = ""
    with reg_out: clear_output()

btn_reg.on_click(on_register)
btn_clear.on_click(on_clear)

display(widgets.HBox([btn_reg, btn_clear]))
display(reg_out)


Output(layout=Layout(border_bottom='1px solid #ddd', border_left='1px solid #ddd', border_right='1px solid #dd…

In [9]:

display(styled_html("📋 STUDENT DIRECTORY", THEME['dark']))

view_dept_filter = make_dropdown("Filter Dept", ["All"] + DEPARTMENTS)
view_sem_filter  = make_dropdown("Filter Sem",  ["All"] + SEMESTERS)
btn_view_all     = make_btn("🔍 Load Students", THEME['primary'])
view_out         = make_output()

def load_students(b):
    with view_out:
        clear_output()
        sql = "SELECT student_id, name, department, semester, gender, email, status FROM students WHERE 1=1"
        params = []
        if view_dept_filter.value != "All":
            sql += " AND department=?"; params.append(view_dept_filter.value)
        if view_sem_filter.value != "All":
            sql += " AND semester=?";  params.append(view_sem_filter.value)
        sql += " ORDER BY name"
        df = query_df(sql, params)
        if df.empty:
            display(HTML('<p style="color:orange">⚠ No students found.</p>'))
        else:
            display(HTML(f'<b>Total: {len(df)} students</b>'))
            display(df.style.set_properties(**{"text-align":"left"})
                          .set_table_styles([{"selector":"th","props":[("background","#4A90D9"),("color","white")]}]))

btn_view_all.on_click(load_students)
display(widgets.HBox([view_dept_filter, view_sem_filter, btn_view_all]))
display(view_out)

Output(layout=Layout(border_bottom='1px solid #ddd', border_left='1px solid #ddd', border_right='1px solid #dd…

In [10]:

display(styled_html("🔎 DYNAMIC STUDENT SEARCH", THEME['dark']))

search_input = widgets.Text(description="Search:", placeholder="Name / ID / Email / Department",
                            layout=widgets.Layout(width="500px"), style={"description_width":"80px"})
btn_search   = make_btn("Search", THEME['primary'])
search_out   = make_output()

def on_search(b):
    with search_out:
        clear_output()
        q = f"%{search_input.value.strip()}%"
        sql = """SELECT student_id, name, department, semester, email, phone
                 FROM students
                 WHERE name LIKE ? OR student_id LIKE ? OR email LIKE ? OR department LIKE ?
                 ORDER BY name"""
        df = query_df(sql, (q, q, q, q))
        if df.empty:
            display(HTML('<p style="color:orange">⚠ No results found.</p>'))
        else:
            display(HTML(f'<b>{len(df)} result(s):</b>'))
            display(df)

btn_search.on_click(on_search)
display(widgets.HBox([search_input, btn_search]))
display(search_out)


Output(layout=Layout(border_bottom='1px solid #ddd', border_left='1px solid #ddd', border_right='1px solid #dd…

In [11]:

display(styled_html("✏️ UPDATE STUDENT", THEME['warning']))

labels, ids = get_student_ids()
upd_select = make_dropdown("Select Student", labels if labels else ["No students"])
upd_field  = make_dropdown("Field to Update",["name","phone","address","email","semester","status"])
upd_value  = make_text("New Value", ph="Enter updated value")
btn_update = make_btn("💾 Update", THEME['warning'])
upd_out    = make_output()

def on_update(b):
    with upd_out:
        clear_output()
        idx = labels.index(upd_select.value) if upd_select.value in labels else -1
        if idx < 0 or upd_value.value.strip() == "":
            display(HTML('<p style="color:red">❌ Select student and enter value.</p>')); return
        sid = ids[idx]
        field = upd_field.value
        execute_query(f"UPDATE students SET {field}=? WHERE student_id=?",
                      (upd_value.value.strip(), sid))
        log_action(f"Updated {field} for {sid}")
        display(HTML(f'<p style="color:green">✅ {field} updated for {sid}</p>'))

btn_update.on_click(on_update)
display(widgets.VBox([upd_select, upd_field, upd_value, btn_update, upd_out]))

In [12]:

display(styled_html("🗑️ DELETE STUDENT", THEME['danger']))

labels2, ids2 = get_student_ids()
del_select   = make_dropdown("Select Student", labels2 if labels2 else ["No students"])
del_confirm  = widgets.Checkbox(description="Confirm deletion", value=False)
btn_delete   = make_btn("❌ Delete", THEME['danger'])
del_out      = make_output()

def on_delete(b):
    with del_out:
        clear_output()
        if not del_confirm.value:
            display(HTML('<p style="color:orange">⚠ Check confirm checkbox first.</p>')); return
        idx = labels2.index(del_select.value) if del_select.value in labels2 else -1
        if idx < 0: return
        sid = ids2[idx]
        execute_query("DELETE FROM attendance WHERE student_id=?", (sid,))
        execute_query("DELETE FROM marks WHERE student_id=?", (sid,))
        execute_query("DELETE FROM students WHERE student_id=?", (sid,))
        log_action(f"Deleted student: {sid}")
        display(HTML(f'<p style="color:red">🗑 Student {sid} deleted permanently.</p>'))
        del_confirm.value = False

btn_delete.on_click(on_delete)
display(widgets.VBox([del_select, del_confirm, btn_delete, del_out]))

In [13]:

display(styled_html("🪪 STUDENT PROFILE VIEWER", THEME['primary']))

labels3, ids3 = get_student_ids()
prof_select  = make_dropdown("Select Student", labels3)
btn_profile  = make_btn("👁 View Profile", THEME['primary'])
prof_out     = make_output()

def view_profile(b):
    with prof_out:
        clear_output()
        idx = labels3.index(prof_select.value)
        sid = ids3[idx]
        df  = query_df("SELECT * FROM students WHERE student_id=?", (sid,))
        if df.empty: return
        row = df.iloc[0]
        html = f"""<table style="border-collapse:collapse;width:500px">
        {''.join(f'<tr><td style="padding:6px 12px;background:#4A90D9;color:white;font-weight:bold">{col}</td><td style="padding:6px 12px;border-bottom:1px solid #ddd">{row[col]}</td></tr>' for col in df.columns)}
        </table>"""
        display(HTML(html))

btn_profile.on_click(view_profile)
display(widgets.HBox([prof_select, btn_profile]))
display(prof_out)

Output(layout=Layout(border_bottom='1px solid #ddd', border_left='1px solid #ddd', border_right='1px solid #dd…

In [14]:

display(styled_html("📅 MARK ATTENDANCE", THEME['success']))

labels4, ids4 = get_student_ids()
att_student = make_dropdown("Student",  labels4)
att_dept_dd = make_dropdown("Dept",     DEPARTMENTS)
att_subject = make_dropdown("Subject",  SUBJECTS_MAP["Computer Science"])
att_date    = make_text("Date", val=str(date.today()), ph="YYYY-MM-DD")
att_status  = make_dropdown("Status",   ["Present","Absent","Late","Medical Leave"])
btn_mark    = make_btn("✅ Mark", THEME['success'])
att_out     = make_output()

def update_subjects(change):
    att_subject.options = SUBJECTS_MAP.get(att_dept_dd.value, [])

att_dept_dd.observe(update_subjects, names='value')

def on_mark(b):
    with att_out:
        clear_output()
        idx = labels4.index(att_student.value)
        sid = ids4[idx]
        try:
            execute_query("INSERT INTO attendance (student_id,subject,date,status) VALUES (?,?,?,?)",
                          (sid, att_subject.value, att_date.value, att_status.value))
            display(HTML(f'<p style="color:green">✅ Attendance marked: {att_student.value} | {att_subject.value} | {att_status.value}</p>'))
        except Exception as e:
            display(HTML(f'<p style="color:red">❌ {e}</p>'))

btn_mark.on_click(on_mark)
display(widgets.HBox([
    widgets.VBox([att_student, att_dept_dd, att_subject]),
    widgets.VBox([att_date, att_status, btn_mark])
]))
display(att_out)

Output(layout=Layout(border_bottom='1px solid #ddd', border_left='1px solid #ddd', border_right='1px solid #dd…

In [15]:

display(styled_html("📊 VIEW ATTENDANCE RECORD", THEME['primary']))

labels5, ids5 = get_student_ids()
va_student = make_dropdown("Student",  labels5)
va_subject = make_dropdown("Subject",  ["All"] + SUBJECTS_MAP["Computer Science"])
btn_va     = make_btn("🔍 View", THEME['primary'])
va_out     = make_output()

def view_attendance(b):
    with va_out:
        clear_output()
        idx = labels5.index(va_student.value)
        sid = ids5[idx]
        sql = "SELECT subject, date, status FROM attendance WHERE student_id=?"
        params = [sid]
        if va_subject.value != "All":
            sql += " AND subject=?"; params.append(va_subject.value)
        sql += " ORDER BY date DESC"
        df = query_df(sql, params)
        if df.empty:
            display(HTML('<p style="color:orange">No records.</p>'))
        else:
            total   = len(df)
            present = len(df[df['status']=='Present'])
            pct     = round(present/total*100, 1)
            color   = "green" if pct >= 75 else "red"
            display(HTML(f'<b>Total: {total} | Present: {present} | <span style="color:{color}">Attendance: {pct}%</span></b>'))
            display(df.tail(20))

btn_va.on_click(view_attendance)
display(widgets.HBox([va_student, va_subject, btn_va]))
display(va_out)

Output(layout=Layout(border_bottom='1px solid #ddd', border_left='1px solid #ddd', border_right='1px solid #dd…

In [16]:

display(styled_html("📈 ATTENDANCE SUMMARY — ALL STUDENTS", THEME['dark']))

btn_att_summary = make_btn("📊 Generate", THEME['dark'])
att_sum_out     = make_output()

def gen_att_summary(b):
    with att_sum_out:
        clear_output()
        sql = """
        SELECT s.name, a.subject,
               COUNT(*) as total,
               SUM(CASE WHEN a.status='Present' THEN 1 ELSE 0 END) as present,
               ROUND(100.0*SUM(CASE WHEN a.status='Present' THEN 1 ELSE 0 END)/COUNT(*),1) as pct
        FROM attendance a
        JOIN students s ON a.student_id = s.student_id
        GROUP BY a.student_id, a.subject
        ORDER BY s.name, a.subject
        """
        df = query_df(sql)
        def color_pct(val):
            color = 'green' if val >= 75 else 'orange' if val >= 60 else 'red'
            return f'color: {color}; font-weight: bold'
        display(df.style.applymap(color_pct, subset=['pct']))

btn_att_summary.on_click(gen_att_summary)
display(btn_att_summary)
display(att_sum_out)

Button(description='📊 Generate', layout=Layout(height='36px', width='180px'), style=ButtonStyle(button_color='…

Output(layout=Layout(border_bottom='1px solid #ddd', border_left='1px solid #ddd', border_right='1px solid #dd…

In [17]:

display(styled_html("📊 ATTENDANCE CHART", THEME['primary']))

labels6, ids6  = get_student_ids()
chart_student  = make_dropdown("Student", labels6)
btn_att_chart  = make_btn("📊 Plot Chart", THEME['primary'])
att_chart_out  = make_output()

def plot_att_chart(b):
    with att_chart_out:
        clear_output()
        idx = labels6.index(chart_student.value)
        sid = ids6[idx]
        sql = """
        SELECT subject,
               ROUND(100.0*SUM(CASE WHEN status='Present' THEN 1 ELSE 0 END)/COUNT(*),1) as pct
        FROM attendance WHERE student_id=?
        GROUP BY subject
        """
        df = query_df(sql, (sid,))
        if df.empty: display(HTML('<p>No data.</p>')); return
        fig, ax = plt.subplots(figsize=(10,4))
        colors = ['#27AE60' if p >= 75 else '#E74C3C' for p in df['pct']]
        bars = ax.bar(df['subject'], df['pct'], color=colors, edgecolor='white', linewidth=1.2)
        ax.axhline(75, color='black', linestyle='--', linewidth=1.5, label='75% threshold')
        ax.set_ylim(0, 110)
        ax.set_ylabel("Attendance %")
        ax.set_title(f"Attendance — {chart_student.value}", fontsize=13, fontweight='bold')
        for bar, val in zip(bars, df['pct']):
            ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+1.5, f"{val}%",
                    ha='center', va='bottom', fontsize=9)
        ax.legend(); plt.xticks(rotation=30, ha='right'); plt.tight_layout(); plt.show()

btn_att_chart.on_click(plot_att_chart)
display(widgets.HBox([chart_student, btn_att_chart]))
display(att_chart_out)


Output(layout=Layout(border_bottom='1px solid #ddd', border_left='1px solid #ddd', border_right='1px solid #dd…

In [18]:

display(styled_html("📝 ENTER MARKS", THEME['warning']))

labels7, ids7  = get_student_ids()
mk_student  = make_dropdown("Student",  labels7)
mk_dept_dd  = make_dropdown("Dept",     DEPARTMENTS)
mk_subject  = make_dropdown("Subject",  SUBJECTS_MAP["Computer Science"])
mk_semester = make_dropdown("Semester", SEMESTERS)
mk_internal = widgets.FloatSlider(description="Internal(20)", min=0, max=20, step=0.5, value=0,
                                   layout=widgets.Layout(width="380px"),
                                   style={"description_width":"130px"})
mk_midterm  = widgets.FloatSlider(description="Midterm(30)",  min=0, max=30, step=0.5, value=0,
                                   layout=widgets.Layout(width="380px"),
                                   style={"description_width":"130px"})
mk_external = widgets.FloatSlider(description="External(50)", min=0, max=50, step=0.5, value=0,
                                   layout=widgets.Layout(width="380px"),
                                   style={"description_width":"130px"})
btn_save_marks = make_btn("💾 Save Marks", THEME['warning'])
mk_out         = make_output()

def update_mk_subjects(change):
    mk_subject.options = SUBJECTS_MAP.get(mk_dept_dd.value, [])

mk_dept_dd.observe(update_mk_subjects, names='value')

def calc_grade(total):
    if total>=90: return "O"
    elif total>=80: return "A+"
    elif total>=70: return "A"
    elif total>=60: return "B+"
    elif total>=50: return "B"
    elif total>=40: return "C"
    else: return "F"

def save_marks(b):
    with mk_out:
        clear_output()
        idx = labels7.index(mk_student.value)
        sid = ids7[idx]
        i, m, e = mk_internal.value, mk_midterm.value, mk_external.value
        total = round(i*0.2 + m*0.3 + e*0.5, 2)
        grade = calc_grade(total)
        execute_query("""INSERT OR REPLACE INTO marks
            (student_id, subject, semester, internal, midterm, external, grade)
            VALUES (?,?,?,?,?,?,?)""",
            (sid, mk_subject.value, mk_semester.value, i, m, e, grade))
        log_action(f"Marks saved for {sid}: {mk_subject.value}")
        display(HTML(f"""<p style="color:green">✅ Saved | Total: <b>{total}</b> | Grade: <b style="color:#E74C3C">{grade}</b></p>"""))

btn_save_marks.on_click(save_marks)
display(widgets.HBox([
    widgets.VBox([mk_student, mk_dept_dd, mk_subject, mk_semester]),
    widgets.VBox([mk_internal, mk_midterm, mk_external, btn_save_marks])
]))
display(mk_out)

Output(layout=Layout(border_bottom='1px solid #ddd', border_left='1px solid #ddd', border_right='1px solid #dd…

In [19]:

display(styled_html("📋 MARKS SCORECARD", THEME['primary']))

labels8, ids8  = get_student_ids()
sc_student  = make_dropdown("Student",  labels8)
sc_semester = make_dropdown("Semester", ["All"] + SEMESTERS)
btn_scorecard = make_btn("📋 Show", THEME['primary'])
sc_out        = make_output()

def show_scorecard(b):
    with sc_out:
        clear_output()
        idx = labels8.index(sc_student.value)
        sid = ids8[idx]
        sql = """SELECT subject, semester, internal, midterm, external, total, grade
                 FROM marks WHERE student_id=?"""
        params = [sid]
        if sc_semester.value != "All":
            sql += " AND semester=?"; params.append(sc_semester.value)
        sql += " ORDER BY semester, subject"
        df = query_df(sql, params)
        if df.empty:
            display(HTML('<p style="color:orange">No marks found.</p>')); return
        avg_total = round(df['total'].mean(), 2)
        display(HTML(f'<b>Average Total: {avg_total} / 100</b>'))
        def grade_color(val):
            c = {"O":"#27AE60","A+":"#2ECC71","A":"#3498DB","B+":"#F39C12","B":"#E67E22","C":"#E74C3C","F":"#C0392B"}
            return f'background-color:{c.get(val,"white")};color:white;font-weight:bold'
        display(df.style.applymap(grade_color, subset=['grade']))

btn_scorecard.on_click(show_scorecard)
display(widgets.HBox([sc_student, sc_semester, btn_scorecard]))
display(sc_out)

Output(layout=Layout(border_bottom='1px solid #ddd', border_left='1px solid #ddd', border_right='1px solid #dd…

In [20]:

display(styled_html("🕸 MARKS RADAR CHART", THEME['primary']))

labels9, ids9  = get_student_ids()
radar_student  = make_dropdown("Student", labels9)
btn_radar      = make_btn("🕸 Radar", THEME['primary'])
radar_out      = make_output()

def plot_radar(b):
    with radar_out:
        clear_output()
        idx = labels9.index(radar_student.value)
        sid = ids9[idx]
        df = query_df("SELECT subject, total FROM marks WHERE student_id=?", (sid,))
        if df.empty or len(df) < 3: display(HTML('<p>Insufficient data.</p>')); return
        N = len(df)
        cats = list(df['subject'])
        vals = list(df['total'])
        angles = np.linspace(0, 2*np.pi, N, endpoint=False).tolist()
        vals   += vals[:1]; angles += angles[:1]
        fig, ax = plt.subplots(figsize=(6,6), subplot_kw=dict(polar=True))
        ax.plot(angles, vals, 'o-', linewidth=2, color='#4A90D9')
        ax.fill(angles, vals, alpha=0.25, color='#4A90D9')
        ax.set_thetagrids(np.degrees(angles[:-1]), cats)
        ax.set_ylim(0,100)
        ax.set_title(f"Performance Radar — {radar_student.value}", fontsize=12, fontweight='bold', pad=18)
        plt.tight_layout(); plt.show()

btn_radar.on_click(plot_radar)
display(widgets.HBox([radar_student, btn_radar]))
display(radar_out)

Output(layout=Layout(border_bottom='1px solid #ddd', border_left='1px solid #ddd', border_right='1px solid #dd…

In [21]:

display(styled_html("🎯 GPA CALCULATOR", THEME['success']))

CREDITS = {"Data Structures":4,"Algorithms":4,"DBMS":3,"OS":3,"Networks":3,"AI":3,
           "Circuits":4,"Signals":3,"Microprocessors":4,"VLSI":3,"Communication":3,"Control Systems":3,
           "Thermodynamics":4,"Fluid Mechanics":4,"Manufacturing":3,"CAD":3,"Dynamics":3,"Materials":3,
           "Structures":4,"Soil Mech":3,"Survey":3,"Hydraulics":4,"Concrete":3,"Transportation":3,
           "Accounting":3,"Marketing":3,"Finance":4,"HRM":3,"Strategy":3,"Economics":4}

labels10, ids10 = get_student_ids()
gpa_student     = make_dropdown("Student",  labels10)
gpa_semester    = make_dropdown("Semester", ["All"] + SEMESTERS)
btn_gpa         = make_btn("🎯 Calculate GPA", THEME['success'])
gpa_out         = make_output()

def calc_gpa(b):
    with gpa_out:
        clear_output()
        idx = labels10.index(gpa_student.value)
        sid = ids10[idx]
        sql = "SELECT subject, grade FROM marks WHERE student_id=?"
        params = [sid]
        if gpa_semester.value != "All":
            sql += " AND semester=?"; params.append(gpa_semester.value)
        df = query_df(sql, params)
        if df.empty: display(HTML('<p>No marks.</p>')); return
        rows, total_pts, total_creds = [], 0, 0
        for _, row in df.iterrows():
            pts  = GRADE_SCALE.get(row['grade'], 0)
            cred = CREDITS.get(row['subject'], 3)
            total_pts  += pts * cred
            total_creds += cred
            rows.append({"Subject":row['subject'],"Grade":row['grade'],"Points":pts,"Credits":cred,"Weighted":pts*cred})
        gpa = round(total_pts / total_creds, 2) if total_creds else 0
        result_df = pd.DataFrame(rows)
        result_df.loc["Total"] = ["","","",total_creds, total_pts]
        color = "#27AE60" if gpa>=8 else "#F39C12" if gpa>=6 else "#E74C3C"
        display(HTML(f'<h3 style="color:{color}">📊 SGPA: {gpa} / 10.0</h3>'))
        display(result_df)

btn_gpa.on_click(calc_gpa)
display(widgets.HBox([gpa_student, gpa_semester, btn_gpa]))
display(gpa_out)

Output(layout=Layout(border_bottom='1px solid #ddd', border_left='1px solid #ddd', border_right='1px solid #dd…

In [22]:

display(styled_html("📈 GPA TREND ACROSS SEMESTERS", THEME['primary']))

labels11, ids11 = get_student_ids()
trend_student   = make_dropdown("Student", labels11)
btn_trend       = make_btn("📈 Plot Trend", THEME['primary'])
trend_out       = make_output()

def plot_gpa_trend(b):
    with trend_out:
        clear_output()
        idx = labels11.index(trend_student.value)
        sid = ids11[idx]
        sems = SEMESTERS
        gpas = []
        for sem in sems:
            df = query_df("SELECT subject, grade FROM marks WHERE student_id=? AND semester=?", (sid, sem))
            if df.empty: gpas.append(None); continue
            pts, creds = 0, 0
            for _, row in df.iterrows():
                c = CREDITS.get(row['subject'], 3)
                pts += GRADE_SCALE.get(row['grade'], 0) * c; creds += c
            gpas.append(round(pts/creds, 2) if creds else None)

        valid = [(s, g) for s, g in zip(sems, gpas) if g is not None]
        if not valid: display(HTML('<p>No data.</p>')); return
        sv, gv = zip(*valid)
        fig, ax = plt.subplots(figsize=(9,4))
        ax.plot(sv, gv, 'o-', color='#4A90D9', linewidth=2.5, markersize=8)
        ax.fill_between(sv, gv, alpha=0.15, color='#4A90D9')
        ax.axhline(7, color='green', linestyle='--', linewidth=1, label='Good GPA (7.0)')
        for x, y in zip(sv, gv):
            ax.annotate(str(y), (x, y), textcoords="offset points", xytext=(0,10), ha='center')
        ax.set_ylim(0, 10.5); ax.set_ylabel("SGPA")
        ax.set_title(f"GPA Trend — {trend_student.value}", fontweight='bold')
        ax.legend(); plt.tight_layout(); plt.show()

btn_trend.on_click(plot_gpa_trend)
display(widgets.HBox([trend_student, btn_trend]))
display(trend_out)


Output(layout=Layout(border_bottom='1px solid #ddd', border_left='1px solid #ddd', border_right='1px solid #dd…

In [23]:

display(styled_html("🏫 DEPARTMENT GPA COMPARISON", THEME['dark']))

btn_dept_gpa = make_btn("📊 Generate", THEME['dark'])
dept_gpa_out = make_output()

def dept_gpa_comparison(b):
    with dept_gpa_out:
        clear_output()
        data = {}
        for dept in DEPARTMENTS:
            students = query_df("SELECT student_id FROM students WHERE department=?", (dept,))
            all_pts, all_creds = 0, 0
            for _, row in students.iterrows():
                df = query_df("SELECT subject, grade FROM marks WHERE student_id=?", (row['student_id'],))
                for _, mr in df.iterrows():
                    c = CREDITS.get(mr['subject'], 3)
                    all_pts  += GRADE_SCALE.get(mr['grade'], 0) * c
                    all_creds += c
            data[dept] = round(all_pts/all_creds, 2) if all_creds else 0

        fig, ax = plt.subplots(figsize=(10,5))
        depts, gpas = zip(*sorted(data.items(), key=lambda x:-x[1]))
        colors = plt.cm.RdYlGn([g/10 for g in gpas])
        bars = ax.bar(depts, gpas, color=colors, edgecolor='white')
        for bar, g in zip(bars, gpas):
            ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.05,
                    str(g), ha='center', fontweight='bold')
        ax.set_ylim(0, 11); ax.set_ylabel("Average SGPA")
        ax.set_title("Average GPA by Department", fontsize=13, fontweight='bold')
        plt.tight_layout(); plt.show()

btn_dept_gpa.on_click(dept_gpa_comparison)
display(btn_dept_gpa)
display(dept_gpa_out)

Button(description='📊 Generate', layout=Layout(height='36px', width='180px'), style=ButtonStyle(button_color='…

Output(layout=Layout(border_bottom='1px solid #ddd', border_left='1px solid #ddd', border_right='1px solid #dd…

In [24]:

display(styled_html("🥧 GRADE DISTRIBUTION PIE CHART", THEME['primary']))

btn_grade_pie = make_btn("🥧 Generate", THEME['primary'])
pie_out       = make_output()

def grade_pie(b):
    with pie_out:
        clear_output()
        df = query_df("SELECT grade, COUNT(*) as cnt FROM marks GROUP BY grade ORDER BY grade")
        if df.empty: return
        colors_map = {"O":"#27AE60","A+":"#2ECC71","A":"#3498DB","B+":"#F39C12",
                      "B":"#E67E22","C":"#E74C3C","F":"#C0392B"}
        cols = [colors_map.get(g, "grey") for g in df['grade']]
        fig, ax = plt.subplots(figsize=(7,6))
        wedges, texts, autotexts = ax.pie(
            df['cnt'], labels=df['grade'], colors=cols,
            autopct='%1.1f%%', startangle=90,
            wedgeprops=dict(edgecolor='white', linewidth=1.5))
        ax.set_title("Overall Grade Distribution", fontsize=13, fontweight='bold')
        plt.tight_layout(); plt.show()

btn_grade_pie.on_click(grade_pie)
display(btn_grade_pie)
display(pie_out)

Button(description='🥧 Generate', layout=Layout(height='36px', width='180px'), style=ButtonStyle(button_color='…

Output(layout=Layout(border_bottom='1px solid #ddd', border_left='1px solid #ddd', border_right='1px solid #dd…

In [25]:

display(styled_html("🏆 TOP PERFORMERS LEADERBOARD", THEME['warning']))

top_n_slider  = widgets.IntSlider(description="Top N:", min=1, max=20, value=5,
                                   layout=widgets.Layout(width="380px"),
                                   style={"description_width":"80px"})
btn_leaderboard = make_btn("🏆 Generate", THEME['warning'])
lb_out          = make_output()

def show_leaderboard(b):
    with lb_out:
        clear_output()
        sql = """
        SELECT s.student_id, s.name, s.department, s.semester,
               ROUND(AVG(m.total),2) as avg_marks,
               COUNT(m.subject) as subjects
        FROM marks m JOIN students s ON m.student_id=s.student_id
        GROUP BY m.student_id
        ORDER BY avg_marks DESC LIMIT ?
        """
        df = query_df(sql, (top_n_slider.value,))
        df.index = range(1, len(df)+1)
        df.index.name = "Rank"
        medals = {1:"🥇", 2:"🥈", 3:"🥉"}
        df.insert(0, "🏅", [medals.get(i, "⭐") for i in df.index])
        display(df.style.background_gradient(subset=['avg_marks'], cmap='YlOrRd'))

btn_leaderboard.on_click(show_leaderboard)
display(widgets.HBox([top_n_slider, btn_leaderboard]))
display(lb_out)

Output(layout=Layout(border_bottom='1px solid #ddd', border_left='1px solid #ddd', border_right='1px solid #dd…

In [26]:

display(styled_html("🌡️ MARKS HEATMAP", THEME['dark']))

btn_heatmap = make_btn("🌡️ Generate", THEME['dark'])
hm_out      = make_output()

def gen_heatmap(b):
    with hm_out:
        clear_output()
        df = query_df("""
            SELECT s.name, m.subject, m.total
            FROM marks m JOIN students s ON m.student_id=s.student_id
        """)
        if df.empty: return
        pivot = df.pivot_table(index='name', columns='subject', values='total', aggfunc='mean')
        fig, ax = plt.subplots(figsize=(14,6))
        im = ax.imshow(pivot.values, aspect='auto', cmap='RdYlGn', vmin=0, vmax=100)
        ax.set_xticks(range(len(pivot.columns))); ax.set_xticklabels(pivot.columns, rotation=45, ha='right', fontsize=9)
        ax.set_yticks(range(len(pivot.index)));   ax.set_yticklabels(pivot.index, fontsize=10)
        for i in range(len(pivot.index)):
            for j in range(len(pivot.columns)):
                val = pivot.values[i,j]
                if not np.isnan(val):
                    ax.text(j, i, f"{val:.0f}", ha='center', va='center', fontsize=8,
                            color='white' if val < 60 else 'black')
        plt.colorbar(im, ax=ax, label="Total Marks"); ax.set_title("Marks Heatmap", fontsize=13, fontweight='bold')
        plt.tight_layout(); plt.show()

btn_heatmap.on_click(gen_heatmap)
display(btn_heatmap)
display(hm_out)

Button(description='🌡️ Generate', layout=Layout(height='36px', width='180px'), style=ButtonStyle(button_color=…

Output(layout=Layout(border_bottom='1px solid #ddd', border_left='1px solid #ddd', border_right='1px solid #dd…

In [27]:

display(styled_html("📦 SCORE BOX PLOT BY DEPARTMENT", THEME['primary']))

btn_box = make_btn("📦 Plot", THEME['primary'])
box_out = make_output()

def gen_box(b):
    with box_out:
        clear_output()
        df = query_df("""SELECT s.department, m.total
                         FROM marks m JOIN students s ON m.student_id=s.student_id""")
        if df.empty: return
        groups = [df[df['department']==d]['total'].dropna().values for d in DEPARTMENTS if not df[df['department']==d].empty]
        labels_b = [d for d in DEPARTMENTS if not df[df['department']==d].empty]
        fig, ax = plt.subplots(figsize=(10,5))
        bp = ax.boxplot(groups, labels=labels_b, patch_artist=True, notch=True)
        colors = ['#4A90D9','#27AE60','#E74C3C','#F39C12','#9B59B6']
        for patch, c in zip(bp['boxes'], colors): patch.set_facecolor(c); patch.set_alpha(0.7)
        ax.set_ylabel("Total Marks"); ax.set_title("Score Distribution by Department", fontsize=13, fontweight='bold')
        ax.yaxis.grid(True, linestyle='--', alpha=0.7); plt.tight_layout(); plt.show()

btn_box.on_click(gen_box)
display(btn_box)
display(box_out)

Button(description='📦 Plot', layout=Layout(height='36px', width='180px'), style=ButtonStyle(button_color='#4A9…

Output(layout=Layout(border_bottom='1px solid #ddd', border_left='1px solid #ddd', border_right='1px solid #dd…

In [28]:

display(styled_html("🔵 ATTENDANCE vs PERFORMANCE SCATTER", THEME['success']))

btn_scatter = make_btn("🔵 Plot Scatter", THEME['success'])
sc_out2     = make_output()

def gen_scatter(b):
    with sc_out2:
        clear_output()
        att_df = query_df("""
            SELECT student_id,
                   ROUND(100.0*SUM(CASE WHEN status='Present' THEN 1 ELSE 0 END)/COUNT(*),1) as att_pct
            FROM attendance GROUP BY student_id
        """)
        marks_df = query_df("SELECT student_id, ROUND(AVG(total),2) as avg_total FROM marks GROUP BY student_id")
        merged = att_df.merge(marks_df, on='student_id')
        stu_df = query_df("SELECT student_id, name, department FROM students")
        merged = merged.merge(stu_df, on='student_id')
        fig, ax = plt.subplots(figsize=(9,5))
        for dept, grp in merged.groupby('department'):
            ax.scatter(grp['att_pct'], grp['avg_total'], label=dept, s=100, alpha=0.8)
        for _, row in merged.iterrows():
            ax.annotate(row['name'].split()[0], (row['att_pct'], row['avg_total']),
                        fontsize=8, xytext=(3,3), textcoords='offset points')
        z = np.polyfit(merged['att_pct'], merged['avg_total'], 1)
        p = np.poly1d(z)
        xs = np.linspace(merged['att_pct'].min(), merged['att_pct'].max(), 100)
        ax.plot(xs, p(xs), 'k--', linewidth=1.5, alpha=0.6, label='Trend')
        ax.set_xlabel("Attendance %"); ax.set_ylabel("Avg Marks")
        ax.set_title("Attendance vs Academic Performance", fontsize=12, fontweight='bold')
        ax.legend(fontsize=8); plt.tight_layout(); plt.show()

btn_scatter.on_click(gen_scatter)
display(btn_scatter)
display(sc_out2)

Button(description='🔵 Plot Scatter', layout=Layout(height='36px', width='180px'), style=ButtonStyle(button_col…

Output(layout=Layout(border_bottom='1px solid #ddd', border_left='1px solid #ddd', border_right='1px solid #dd…

In [29]:

display(styled_html("📊 FULL ANALYTICS DASHBOARD", THEME['dark']))

btn_dashboard = make_btn("🚀 Launch Dashboard", THEME['dark'])
dash_out      = make_output()

def launch_dashboard(b):
    with dash_out:
        clear_output()
        fig = plt.figure(figsize=(16, 12))
        fig.suptitle("🎓 Student ERP — Analytics Dashboard", fontsize=16, fontweight='bold', y=0.98)
        gs = gridspec.GridSpec(3, 3, figure=fig, hspace=0.5, wspace=0.4)

        # 1: Grade Distribution
        ax1 = fig.add_subplot(gs[0, 0])
        df_g = query_df("SELECT grade, COUNT(*) as cnt FROM marks GROUP BY grade")
        colors_g = ["#27AE60","#2ECC71","#3498DB","#F39C12","#E67E22","#E74C3C","#C0392B"]
        ax1.pie(df_g['cnt'], labels=df_g['grade'], colors=colors_g[:len(df_g)],
                autopct='%1.0f%%', startangle=90, textprops={'fontsize':8})
        ax1.set_title("Grade Distribution", fontsize=10, fontweight='bold')

        # 2: Dept enrollment
        ax2 = fig.add_subplot(gs[0, 1])
        df_d = query_df("SELECT department, COUNT(*) as cnt FROM students GROUP BY department")
        ax2.barh(df_d['department'], df_d['cnt'], color=THEME['primary'], alpha=0.8)
        ax2.set_xlabel("Students"); ax2.set_title("Dept Enrollment", fontsize=10, fontweight='bold')

        # 3: Avg marks per subject
        ax3 = fig.add_subplot(gs[0, 2])
        df_m = query_df("SELECT subject, ROUND(AVG(total),1) as avg FROM marks GROUP BY subject ORDER BY avg DESC LIMIT 8")
        ax3.barh(df_m['subject'], df_m['avg'], color=THEME['success'], alpha=0.8)
        ax3.set_xlabel("Avg Marks"); ax3.set_title("Top Subjects Avg", fontsize=10, fontweight='bold')
        ax3.set_xlim(0, 100)

        # 4: Attendance by dept
        ax4 = fig.add_subplot(gs[1, 0:2])
        df_a = query_df("""
            SELECT s.department,
                   ROUND(100.0*SUM(CASE WHEN a.status='Present' THEN 1 ELSE 0 END)/COUNT(*),1) as pct
            FROM attendance a JOIN students s ON a.student_id=s.student_id
            GROUP BY s.department
        """)
        clrs = ['#27AE60' if p >= 75 else '#E74C3C' for p in df_a['pct']]
        ax4.bar(df_a['department'], df_a['pct'], color=clrs, alpha=0.85)
        ax4.axhline(75, color='black', linestyle='--', linewidth=1.2)
        ax4.set_ylim(0, 110); ax4.set_ylabel("Attendance %")
        ax4.set_title("Avg Attendance by Department", fontsize=10, fontweight='bold')

        # 5: Marks distribution histogram
        ax5 = fig.add_subplot(gs[1, 2])
        df_hist = query_df("SELECT total FROM marks")
        ax5.hist(df_hist['total'], bins=15, color=THEME['warning'], edgecolor='white', alpha=0.85)
        ax5.set_xlabel("Total Marks"); ax5.set_ylabel("Frequency")
        ax5.set_title("Marks Distribution", fontsize=10, fontweight='bold')

        # 6: Gender distribution
        ax6 = fig.add_subplot(gs[2, 0])
        df_gen = query_df("SELECT gender, COUNT(*) as cnt FROM students GROUP BY gender")
        ax6.pie(df_gen['cnt'], labels=df_gen['gender'], autopct='%1.0f%%',
                colors=['#4A90D9','#E74C3C','#27AE60'], startangle=90, textprops={'fontsize':9})
        ax6.set_title("Gender Split", fontsize=10, fontweight='bold')

        # 7: Top students bar
        ax7 = fig.add_subplot(gs[2, 1:])
        df_top = query_df("""
            SELECT s.name, ROUND(AVG(m.total),1) as avg
            FROM marks m JOIN students s ON m.student_id=s.student_id
            GROUP BY m.student_id ORDER BY avg DESC LIMIT 6
        """)
        bars = ax7.bar(df_top['name'], df_top['avg'], color=plt.cm.viridis(np.linspace(0.3, 0.9, len(df_top))))
        for bar, v in zip(bars, df_top['avg']):
            ax7.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.3, str(v),
                     ha='center', fontsize=9, fontweight='bold')
        ax7.set_ylim(0, 110); ax7.set_ylabel("Avg Marks")
        ax7.set_title("Top 6 Students by Avg Marks", fontsize=10, fontweight='bold')
        plt.xticks(rotation=15, ha='right', fontsize=9)

        plt.savefig("erp_dashboard.png", dpi=120, bbox_inches='tight')
        plt.show()
        display(HTML('<p style="color:green">✅ Dashboard saved as erp_dashboard.png</p>'))

btn_dashboard.on_click(launch_dashboard)
display(btn_dashboard)
display(dash_out)

Button(description='🚀 Launch Dashboard', layout=Layout(height='36px', width='180px'), style=ButtonStyle(button…

Output(layout=Layout(border_bottom='1px solid #ddd', border_left='1px solid #ddd', border_right='1px solid #dd…

In [30]:

display(styled_html("🔧 ADMIN PANEL — SYSTEM STATISTICS", THEME['danger']))

btn_stats = make_btn("📊 Load Stats", THEME['danger'])
stats_out = make_output()

def load_stats(b):
    with stats_out:
        clear_output()
        total_stu  = query_df("SELECT COUNT(*) as c FROM students").iloc[0]['c']
        total_att  = query_df("SELECT COUNT(*) as c FROM attendance").iloc[0]['c']
        total_marks= query_df("SELECT COUNT(*) as c FROM marks").iloc[0]['c']
        active_stu = query_df("SELECT COUNT(*) as c FROM students WHERE status='Active'").iloc[0]['c']
        avg_marks  = query_df("SELECT ROUND(AVG(total),2) as a FROM marks").iloc[0]['a']
        att_pct    = query_df("""SELECT ROUND(100.0*SUM(CASE WHEN status='Present' THEN 1 ELSE 0 END)/COUNT(*),1) as p FROM attendance""").iloc[0]['p']

        html = f"""<div style="display:flex;flex-wrap:wrap;gap:12px">
        {''.join([
            f'<div style="background:{color};color:white;padding:16px 24px;border-radius:10px;min-width:160px;text-align:center"><div style="font-size:28px;font-weight:bold">{val}</div><div style="font-size:13px;margin-top:4px">{label}</div></div>'
            for val, label, color in [
                (total_stu, "Total Students", "#4A90D9"),
                (active_stu, "Active Students", "#27AE60"),
                (total_att, "Attendance Records", "#9B59B6"),
                (total_marks, "Marks Records", "#F39C12"),
                (avg_marks, "Overall Avg Marks", "#E74C3C"),
                (f"{att_pct}%", "Overall Attendance", "#1ABC9C"),
            ]
        ])}</div>"""
        display(HTML(html))

btn_stats.on_click(load_stats)
display(btn_stats)
display(stats_out)

Button(description='📊 Load Stats', layout=Layout(height='36px', width='180px'), style=ButtonStyle(button_color…

Output(layout=Layout(border_bottom='1px solid #ddd', border_left='1px solid #ddd', border_right='1px solid #dd…

In [31]:

display(styled_html("📅 BULK ATTENDANCE REPORT — LOW ATTENDANCE", THEME['danger']))

threshold_slider = widgets.IntSlider(description="Min. Att%:", min=50, max=90, value=75,
                                     layout=widgets.Layout(width="400px"),
                                     style={"description_width":"110px"})
btn_low_att = make_btn("⚠ Find Low Att.", THEME['danger'])
low_att_out = make_output()

def find_low_att(b):
    with low_att_out:
        clear_output()
        sql = """
        SELECT s.student_id, s.name, s.department, a.subject,
               ROUND(100.0*SUM(CASE WHEN a.status='Present' THEN 1 ELSE 0 END)/COUNT(*),1) as pct
        FROM attendance a JOIN students s ON a.student_id=s.student_id
        GROUP BY a.student_id, a.subject
        HAVING pct < ?
        ORDER BY pct ASC
        """
        df = query_df(sql, (threshold_slider.value,))
        if df.empty:
            display(HTML('<p style="color:green">✅ All students meet the attendance threshold!</p>'))
        else:
            display(HTML(f'<b style="color:red">⚠ {len(df)} cases below {threshold_slider.value}%:</b>'))
            display(df.style.background_gradient(subset=['pct'], cmap='Reds_r'))

btn_low_att.on_click(find_low_att)
display(widgets.HBox([threshold_slider, btn_low_att]))
display(low_att_out)


Output(layout=Layout(border_bottom='1px solid #ddd', border_left='1px solid #ddd', border_right='1px solid #dd…

In [32]:

display(styled_html("❌ FAILED STUDENTS REPORT", THEME['danger']))

btn_failed = make_btn("❌ Show Failed", THEME['danger'])
failed_out = make_output()

def show_failed(b):
    with failed_out:
        clear_output()
        sql = """
        SELECT s.name, s.department, m.subject, m.semester, m.total, m.grade
        FROM marks m JOIN students s ON m.student_id=s.student_id
        WHERE m.grade='F'
        ORDER BY s.name
        """
        df = query_df(sql)
        if df.empty:
            display(HTML('<p style="color:green">✅ No failures recorded!</p>'))
        else:
            display(HTML(f'<b style="color:red">Total Failures: {len(df)}</b>'))
            display(df.style.applymap(lambda v: 'background-color:#FFB3B3' if v=='F' else '', subset=['grade']))

btn_failed.on_click(show_failed)
display(btn_failed)
display(failed_out)


Button(description='❌ Show Failed', layout=Layout(height='36px', width='180px'), style=ButtonStyle(button_colo…

Output(layout=Layout(border_bottom='1px solid #ddd', border_left='1px solid #ddd', border_right='1px solid #dd…

In [33]:

display(styled_html("⬆️ BULK SEMESTER PROMOTION", THEME['success']))

prom_dept   = make_dropdown("Department", ["All"] + DEPARTMENTS)
btn_promote = make_btn("⬆️ Promote All", THEME['success'])
prom_out    = make_output()

SEM_MAP = {"Sem 1":"Sem 2","Sem 2":"Sem 3","Sem 3":"Sem 4",
           "Sem 4":"Sem 5","Sem 5":"Sem 6","Sem 6":"Sem 7","Sem 7":"Sem 8","Sem 8":"Sem 8"}

def promote_students(b):
    with prom_out:
        clear_output()
        if prom_dept.value == "All":
            df = query_df("SELECT student_id, semester FROM students")
        else:
            df = query_df("SELECT student_id, semester FROM students WHERE department=?", (prom_dept.value,))
        count = 0
        for _, row in df.iterrows():
            new_sem = SEM_MAP.get(row['semester'], row['semester'])
            if new_sem != row['semester']:
                execute_query("UPDATE students SET semester=? WHERE student_id=?",
                              (new_sem, row['student_id']))
                count += 1
        log_action(f"Promoted {count} students ({prom_dept.value})")
        display(HTML(f'<p style="color:green">✅ {count} student(s) promoted to next semester.</p>'))

btn_promote.on_click(promote_students)
display(widgets.HBox([prom_dept, btn_promote]))
display(prom_out)

Output(layout=Layout(border_bottom='1px solid #ddd', border_left='1px solid #ddd', border_right='1px solid #dd…

In [34]:

display(styled_html("💾 EXPORT DATA", THEME['dark']))

export_type = make_dropdown("Export", ["Students","Marks","Attendance","Admin Log"])
btn_export  = make_btn("💾 Export CSV", THEME['dark'])
exp_out     = make_output()

TABLE_MAP = {"Students":"students","Marks":"marks","Attendance":"attendance","Admin Log":"admin_log"}

def export_data(b):
    with exp_out:
        clear_output()
        tbl  = TABLE_MAP[export_type.value]
        df   = query_df(f"SELECT * FROM {tbl}")
        fname = f"{tbl}_export.csv"
        df.to_csv(fname, index=False)
        display(HTML(f'<p style="color:green">✅ Exported {len(df)} records → <b>{fname}</b></p>'))
        display(df.head(5))

btn_export.on_click(export_data)
display(widgets.HBox([export_type, btn_export]))
display(exp_out)

Output(layout=Layout(border_bottom='1px solid #ddd', border_left='1px solid #ddd', border_right='1px solid #dd…

In [35]:

display(styled_html("📜 ADMIN ACTIVITY LOG", THEME['dark']))

btn_log = make_btn("📜 View Log", THEME['dark'])
log_out = make_output()

def view_log(b):
    with log_out:
        clear_output()
        df = query_df("SELECT action, performed_by, timestamp FROM admin_log ORDER BY id DESC LIMIT 30")
        if df.empty:
            display(HTML('<p>No log entries.</p>'))
        else:
            display(df)

btn_log.on_click(view_log)
display(btn_log)
display(log_out)


Button(description='📜 View Log', layout=Layout(height='36px', width='180px'), style=ButtonStyle(button_color='…

Output(layout=Layout(border_bottom='1px solid #ddd', border_left='1px solid #ddd', border_right='1px solid #dd…

In [36]:

display(styled_html("⚠️ RESET DATABASE (Danger Zone)", THEME['danger']))

confirm_reset = widgets.Checkbox(description="I understand this will erase all data", value=False)
btn_reset     = make_btn("🔴 RESET ALL", THEME['danger'])
reset_out     = make_output()

def reset_db(b):
    with reset_out:
        clear_output()
        if not confirm_reset.value:
            display(HTML('<p style="color:orange">⚠ Check the confirmation box.</p>')); return
        for tbl in ['admin_log','marks','attendance','students','courses']:
            execute_query(f"DELETE FROM {tbl}")
        display(HTML('<p style="color:red;font-weight:bold">⚠ All data cleared. Re-run Cell 4 to reseed.</p>'))
        confirm_reset.value = False

btn_reset.on_click(reset_db)
display(confirm_reset)
display(btn_reset)
display(reset_out)

Checkbox(value=False, description='I understand this will erase all data')

Button(description='🔴 RESET ALL', layout=Layout(height='36px', width='180px'), style=ButtonStyle(button_color=…

Output(layout=Layout(border_bottom='1px solid #ddd', border_left='1px solid #ddd', border_right='1px solid #dd…

In [37]:

display(styled_html("🖥 SQL QUERY TERMINAL", THEME['dark']))

sql_editor = widgets.Textarea(
    value="SELECT * FROM students;",
    layout=widgets.Layout(width="700px", height="100px"),
    placeholder="Enter SQL query..."
)
btn_run_sql = make_btn("▶ Run SQL", THEME['dark'])
sql_out     = make_output()

def run_sql(b):
    with sql_out:
        clear_output()
        try:
            df = query_df(sql_editor.value)
            display(HTML(f'<b>Rows returned: {len(df)}</b>'))
            display(df)
        except Exception as e:
            display(HTML(f'<p style="color:red">❌ SQL Error: {e}</p>'))

btn_run_sql.on_click(run_sql)
display(sql_editor)
display(btn_run_sql)
display(sql_out)

Textarea(value='SELECT * FROM students;', layout=Layout(height='100px', width='700px'), placeholder='Enter SQL…

Button(description='▶ Run SQL', layout=Layout(height='36px', width='180px'), style=ButtonStyle(button_color='#…

Output(layout=Layout(border_bottom='1px solid #ddd', border_left='1px solid #ddd', border_right='1px solid #dd…

In [38]:

display(styled_html("🏷 PERFORMANCE CATEGORY CLASSIFIER", THEME['success']))

btn_classify = make_btn("🏷 Classify All", THEME['success'])
classify_out = make_output()

def classify_students(b):
    with classify_out:
        clear_output()
        sql = """
        SELECT s.name, s.department,
               ROUND(AVG(m.total),2) as avg_marks,
               ROUND(100.0*SUM(CASE WHEN a.status='Present' THEN 1 ELSE 0 END)/COUNT(DISTINCT a.id),1) as att_pct
        FROM students s
        LEFT JOIN marks m ON s.student_id=m.student_id
        LEFT JOIN attendance a ON s.student_id=a.student_id
        GROUP BY s.student_id
        """
        df = query_df(sql)

        def categorize(row):
            m, a = row['avg_marks'], row['att_pct']
            if m >= 80 and a >= 85: return "🌟 Distinction"
            elif m >= 70 and a >= 75: return "👍 Merit"
            elif m >= 60 and a >= 75: return "✅ Pass"
            elif m < 50 or a < 60: return "🚨 At Risk"
            else: return "⚠ Average"

        df['category'] = df.apply(categorize, axis=1)
        display(df[['name','department','avg_marks','att_pct','category']].sort_values('category'))

btn_classify.on_click(classify_students)
display(btn_classify)
display(classify_out)

Button(description='🏷 Classify All', layout=Layout(height='36px', width='180px'), style=ButtonStyle(button_col…

Output(layout=Layout(border_bottom='1px solid #ddd', border_left='1px solid #ddd', border_right='1px solid #dd…

In [39]:

display(styled_html("🗒 REPORT CARD GENERATOR", THEME['primary']))

labels12, ids12 = get_student_ids()
rc_student  = make_dropdown("Student",  labels12)
btn_rc      = make_btn("🗒 Generate", THEME['primary'])
rc_out      = make_output()

def gen_report_card(b):
    with rc_out:
        clear_output()
        idx = labels12.index(rc_student.value)
        sid = ids12[idx]
        stu = query_df("SELECT * FROM students WHERE student_id=?", (sid,)).iloc[0]
        marks_df = query_df("SELECT subject, internal, midterm, external, total, grade FROM marks WHERE student_id=?", (sid,))
        att_df = query_df("""
            SELECT subject,
                   COUNT(*) as total_classes,
                   SUM(CASE WHEN status='Present' THEN 1 ELSE 0 END) as attended,
                   ROUND(100.0*SUM(CASE WHEN status='Present' THEN 1 ELSE 0 END)/COUNT(*),1) as att_pct
            FROM attendance WHERE student_id=? GROUP BY subject
        """, (sid,))

        avg_m = round(marks_df['total'].mean(), 2) if not marks_df.empty else 0
        avg_a = round(att_df['att_pct'].mean(), 1) if not att_df.empty else 0

        pts, creds = 0, 0
        for _, row in marks_df.iterrows():
            c = CREDITS.get(row['subject'], 3)
            pts += GRADE_SCALE.get(row['grade'], 0) * c; creds += c
        gpa = round(pts/creds, 2) if creds else 0

        display(HTML(f"""
        <div style="border:2px solid #4A90D9;border-radius:12px;padding:20px;max-width:720px;font-family:Arial">
          <div style="background:#4A90D9;color:white;padding:10px 20px;border-radius:8px;text-align:center;font-size:18px;font-weight:bold">
            🎓 STUDENT REPORT CARD</div>
          <table style="width:100%;margin-top:12px;border-collapse:collapse">
            <tr><td style="padding:6px;font-weight:bold;color:#555">Student ID</td><td>{stu['student_id']}</td>
                <td style="font-weight:bold;color:#555">Name</td><td>{stu['name']}</td></tr>
            <tr><td style="padding:6px;font-weight:bold;color:#555">Department</td><td>{stu['department']}</td>
                <td style="font-weight:bold;color:#555">Semester</td><td>{stu['semester']}</td></tr>
            <tr><td style="padding:6px;font-weight:bold;color:#555">Avg Marks</td>
                <td style="font-weight:bold;color:#27AE60">{avg_m}/100</td>
                <td style="font-weight:bold;color:#555">SGPA</td>
                <td style="font-weight:bold;color:#E74C3C;font-size:16px">{gpa}/10</td></tr>
            <tr><td style="padding:6px;font-weight:bold;color:#555">Attendance</td>
                <td style="color:{'green' if avg_a>=75 else 'red'};font-weight:bold">{avg_a}%</td>
                <td style="font-weight:bold;color:#555">Status</td><td>{stu['status']}</td></tr>
          </table>
        </div>
        """))
        display(HTML('<b>📊 Subject-wise Marks:</b>'))
        display(marks_df)
        display(HTML('<b>📅 Attendance Summary:</b>'))
        display(att_df)

btn_rc.on_click(gen_report_card)
display(widgets.HBox([rc_student, btn_rc]))
display(rc_out)

Output(layout=Layout(border_bottom='1px solid #ddd', border_left='1px solid #ddd', border_right='1px solid #dd…

In [40]:

display(styled_html("🎓 STUDENT ERP — TABBED MAIN INTERFACE", THEME['dark']))

# Tab 1: Quick Stats
t1_out = make_output()
with t1_out:
    df_stu = query_df("SELECT * FROM students ORDER BY name")
    display(HTML('<b>📋 Registered Students</b>'))
    display(df_stu[['student_id','name','department','semester','status']])

# Tab 2: Marks Summary
t2_out = make_output()
with t2_out:
    df_mk = query_df("""
        SELECT s.name, m.subject, m.total, m.grade
        FROM marks m JOIN students s ON m.student_id=s.student_id
        ORDER BY s.name, m.subject
    """)
    display(HTML('<b>📝 Marks Overview</b>'))
    display(df_mk)

# Tab 3: Attendance Overview
t3_out = make_output()
with t3_out:
    df_at = query_df("""
        SELECT s.name, a.subject,
               ROUND(100.0*SUM(CASE WHEN a.status='Present' THEN 1 ELSE 0 END)/COUNT(*),1) as att_pct
        FROM attendance a JOIN students s ON a.student_id=s.student_id
        GROUP BY a.student_id, a.subject
    """)
    display(HTML('<b>📅 Attendance Overview</b>'))
    display(df_at.style.background_gradient(subset=['att_pct'], cmap='RdYlGn', vmin=0, vmax=100))

# Tab 4: GPA Summary
t4_out = make_output()
with t4_out:
    rows = []
    for sid, name in zip(ids[:6], [l.split(' - ')[1] for l in labels[:6]]):
        df_gpa = query_df("SELECT subject, grade FROM marks WHERE student_id=?", (sid,))
        pts, creds = 0, 0
        for _, r in df_gpa.iterrows():
            c = CREDITS.get(r['subject'], 3); pts += GRADE_SCALE.get(r['grade'],0)*c; creds += c
        rows.append({"Student":name, "SGPA": round(pts/creds,2) if creds else 0})
    gpa_df = pd.DataFrame(rows).sort_values('SGPA', ascending=False)
    display(HTML('<b>🎯 GPA Summary</b>'))
    display(gpa_df)

tab = widgets.Tab(children=[t1_out, t2_out, t3_out, t4_out])
tab.set_title(0, "👥 Students")
tab.set_title(1, "📝 Marks")
tab.set_title(2, "📅 Attendance")
tab.set_title(3, "🎯 GPA")
display(tab)

In [41]:

display(styled_html("📅 MONTHLY ATTENDANCE TREND", THEME['primary']))

btn_monthly = make_btn("📅 Plot Monthly", THEME['primary'])
monthly_out = make_output()

def plot_monthly(b):
    with monthly_out:
        clear_output()
        df = query_df("""
            SELECT SUBSTR(date,1,7) as month,
                   ROUND(100.0*SUM(CASE WHEN status='Present' THEN 1 ELSE 0 END)/COUNT(*),1) as pct
            FROM attendance GROUP BY month ORDER BY month
        """)
        if df.empty: return
        fig, ax = plt.subplots(figsize=(10,4))
        ax.plot(df['month'], df['pct'], 'o-', color=THEME['primary'], linewidth=2.5, markersize=8)
        ax.fill_between(df['month'], df['pct'], alpha=0.15, color=THEME['primary'])
        ax.axhline(75, color='red', linestyle='--', linewidth=1.3, label='75% Line')
        ax.set_ylabel("Avg Attendance %"); ax.set_xlabel("Month")
        ax.set_title("Monthly Attendance Trend", fontsize=12, fontweight='bold')
        ax.set_ylim(0, 110); ax.legend(); plt.xticks(rotation=30); plt.tight_layout(); plt.show()

btn_monthly.on_click(plot_monthly)
display(btn_monthly)
display(monthly_out)

Button(description='📅 Plot Monthly', layout=Layout(height='36px', width='180px'), style=ButtonStyle(button_col…

Output(layout=Layout(border_bottom='1px solid #ddd', border_left='1px solid #ddd', border_right='1px solid #dd…

In [42]:

display(styled_html("📚 SUBJECT-WISE CLASS AVERAGE", THEME['success']))

subj_dept_filter = make_dropdown("Department", DEPARTMENTS)
btn_subj_avg     = make_btn("📚 Compute", THEME['success'])
subj_out         = make_output()

def subj_avg(b):
    with subj_out:
        clear_output()
        subs = SUBJECTS_MAP[subj_dept_filter.value]
        rows = []
        for sub in subs:
            df = query_df("SELECT AVG(internal) as i, AVG(midterm) as m, AVG(external) as e, AVG(total) as t FROM marks WHERE subject=?", (sub,))
            row = df.iloc[0]
            rows.append({"Subject":sub,
                         "Avg Internal":round(row['i'],1) if row['i'] else 0,
                         "Avg Midterm":round(row['m'],1) if row['m'] else 0,
                         "Avg External":round(row['e'],1) if row['e'] else 0,
                         "Avg Total":round(row['t'],1) if row['t'] else 0})
        result = pd.DataFrame(rows)
        fig, ax = plt.subplots(figsize=(10,4))
        x = range(len(result))
        w = 0.2
        ax.bar([i-w*1.5 for i in x], result['Avg Internal'], w, label='Internal', color='#4A90D9', alpha=0.85)
        ax.bar([i-w*0.5 for i in x], result['Avg Midterm'],  w, label='Midterm',  color='#27AE60', alpha=0.85)
        ax.bar([i+w*0.5 for i in x], result['Avg External'], w, label='External', color='#E74C3C', alpha=0.85)
        ax.bar([i+w*1.5 for i in x], result['Avg Total'],    w, label='Total',    color='#F39C12', alpha=0.85)
        ax.set_xticks(list(x)); ax.set_xticklabels(result['Subject'], rotation=30, ha='right')
        ax.set_ylabel("Average Marks"); ax.set_title(f"Subject Averages — {subj_dept_filter.value}", fontweight='bold')
        ax.legend(); plt.tight_layout(); plt.show()
        display(result)

btn_subj_avg.on_click(subj_avg)
display(widgets.HBox([subj_dept_filter, btn_subj_avg]))
display(subj_out)

Output(layout=Layout(border_bottom='1px solid #ddd', border_left='1px solid #ddd', border_right='1px solid #dd…

In [43]:

display(styled_html("⚖️ STUDENT COMPARISON TOOL", THEME['warning']))

labels_c1 = list(labels); labels_c2 = list(labels)
cmp_stu1 = make_dropdown("Student A", labels_c1)
cmp_stu2 = make_dropdown("Student B", labels_c2)
btn_cmp  = make_btn("⚖️ Compare", THEME['warning'])
cmp_out  = make_output()

def compare_students(b):
    with cmp_out:
        clear_output()
        i1 = labels_c1.index(cmp_stu1.value); i2 = labels_c2.index(cmp_stu2.value)
        s1, s2 = ids[i1], ids[i2]
        df1 = query_df("SELECT subject, total FROM marks WHERE student_id=?", (s1,))
        df2 = query_df("SELECT subject, total FROM marks WHERE student_id=?", (s2,))
        merged = df1.merge(df2, on='subject', suffixes=('_A','_B'))
        if merged.empty: display(HTML('<p>No common subjects.</p>')); return
        fig, ax = plt.subplots(figsize=(10,5))
        x = range(len(merged))
        ax.bar([i-0.2 for i in x], merged['total_A'], 0.4, label=cmp_stu1.value.split(' - ')[1], color='#4A90D9', alpha=0.85)
        ax.bar([i+0.2 for i in x], merged['total_B'], 0.4, label=cmp_stu2.value.split(' - ')[1], color='#E74C3C', alpha=0.85)
        ax.set_xticks(list(x)); ax.set_xticklabels(merged['subject'], rotation=30, ha='right')
        ax.set_ylabel("Total Marks"); ax.set_ylim(0, 110)
        ax.set_title("Head-to-Head Comparison", fontweight='bold')
        ax.legend(); plt.tight_layout(); plt.show()

btn_cmp.on_click(compare_students)
display(widgets.HBox([cmp_stu1, cmp_stu2, btn_cmp]))
display(cmp_out)

Output(layout=Layout(border_bottom='1px solid #ddd', border_left='1px solid #ddd', border_right='1px solid #dd…

In [44]:

display(styled_html("🔄 BATCH MARKS UPDATE", THEME['warning']))

batch_out = make_output()
btn_batch = make_btn("📄 Load Template", THEME['warning'])
btn_apply = make_btn("💾 Apply Updates", THEME['success'])
batch_data = {}

def load_batch_template(b):
    with batch_out:
        clear_output()
        df = query_df("SELECT student_id, subject, internal, midterm, external FROM marks LIMIT 10")
        display(HTML('<i>Edit values in the query terminal (Cell 37) and use the Batch Update workflow.</i>'))
        display(df)
        batch_data['df'] = df
        display(HTML('<p style="color:blue">Template loaded. Modify via SQL terminal then re-run.</p>'))

btn_batch.on_click(load_batch_template)
display(widgets.HBox([btn_batch]))
display(batch_out)

Output(layout=Layout(border_bottom='1px solid #ddd', border_left='1px solid #ddd', border_right='1px solid #dd…

In [45]:

display(styled_html("📊 ALL STUDENTS GPA OVERVIEW", THEME['primary']))

btn_all_gpa = make_btn("📊 Generate", THEME['primary'])
all_gpa_out = make_output()

def all_students_gpa(b):
    with all_gpa_out:
        clear_output()
        rows = []
        for sid, label in zip(ids, labels):
            name = label.split(' - ')[1]
            df_m = query_df("SELECT subject, grade FROM marks WHERE student_id=?", (sid,))
            pts, creds = 0, 0
            for _, r in df_m.iterrows():
                c = CREDITS.get(r['subject'], 3); pts += GRADE_SCALE.get(r['grade'],0)*c; creds += c
            gpa = round(pts/creds,2) if creds else 0
            dept = query_df("SELECT department FROM students WHERE student_id=?", (sid,)).iloc[0]['department']
            rows.append({"Name":name,"Department":dept,"SGPA":gpa,"Credits":creds})
        gdf = pd.DataFrame(rows).sort_values("SGPA", ascending=True)
        fig, ax = plt.subplots(figsize=(9,5))
        colors = [THEME['success'] if g>=8 else THEME['warning'] if g>=6 else THEME['danger'] for g in gdf['SGPA']]
        bars = ax.barh(gdf['Name'], gdf['SGPA'], color=colors, alpha=0.85)
        for bar, g in zip(bars, gdf['SGPA']):
            ax.text(bar.get_width()+0.05, bar.get_y()+bar.get_height()/2, str(g), va='center', fontsize=10)
        ax.set_xlim(0, 11); ax.axvline(8, color='green', linestyle='--', linewidth=1.2, label='8.0 Threshold')
        ax.set_title("All Students SGPA", fontweight='bold'); ax.legend(); plt.tight_layout(); plt.show()

btn_all_gpa.on_click(all_students_gpa)
display(btn_all_gpa)
display(all_gpa_out)

Button(description='📊 Generate', layout=Layout(height='36px', width='180px'), style=ButtonStyle(button_color='…

Output(layout=Layout(border_bottom='1px solid #ddd', border_left='1px solid #ddd', border_right='1px solid #dd…

In [46]:

display(styled_html("📖 COURSE MANAGEMENT", THEME['primary']))

crs_code  = make_text("Course Code", ph="CS301")
crs_name  = make_text("Course Name", ph="Data Structures")
crs_dept  = make_dropdown("Department", DEPARTMENTS)
crs_cred  = widgets.IntSlider(description="Credits:", min=1, max=5, value=3,
                               layout=widgets.Layout(width="380px"),
                               style={"description_width":"130px"})
crs_sem   = make_dropdown("Semester",   SEMESTERS)
btn_add_crs = make_btn("➕ Add Course", THEME['primary'])
crs_out     = make_output()

def add_course(b):
    with crs_out:
        clear_output()
        if not crs_code.value or not crs_name.value:
            display(HTML('<p style="color:red">❌ Code and Name are required.</p>')); return
        try:
            execute_query("INSERT INTO courses (course_code,course_name,department,credits,semester) VALUES (?,?,?,?,?)",
                          (crs_code.value.strip(), crs_name.value.strip(), crs_dept.value, crs_cred.value, crs_sem.value))
            display(HTML(f'<p style="color:green">✅ Course {crs_name.value} added.</p>'))
        except sqlite3.IntegrityError:
            display(HTML('<p style="color:red">❌ Course code already exists.</p>'))

btn_add_crs.on_click(add_course)
display(widgets.HBox([
    widgets.VBox([crs_code, crs_name, crs_dept]),
    widgets.VBox([crs_cred, crs_sem, btn_add_crs])
]))
display(crs_out)


Output(layout=Layout(border_bottom='1px solid #ddd', border_left='1px solid #ddd', border_right='1px solid #dd…

In [47]:

display(styled_html("📚 COURSE CATALOG", THEME['success']))

crs_view_dept = make_dropdown("Filter", ["All"] + DEPARTMENTS)
btn_view_crs  = make_btn("📚 View Courses", THEME['success'])
crs_view_out  = make_output()

def view_courses(b):
    with crs_view_out:
        clear_output()
        sql = "SELECT * FROM courses"
        params = []
        if crs_view_dept.value != "All":
            sql += " WHERE department=?"; params.append(crs_view_dept.value)
        df = query_df(sql, params)
        if df.empty:
            display(HTML('<p>No courses found. Add using Cell 46 or wait — pre-loaded subjects are in SUBJECTS_MAP.</p>'))
        else:
            display(df)

btn_view_crs.on_click(view_courses)
display(widgets.HBox([crs_view_dept, btn_view_crs]))
display(crs_view_out)

Output(layout=Layout(border_bottom='1px solid #ddd', border_left='1px solid #ddd', border_right='1px solid #dd…

In [48]:

display(styled_html("📉 INTERNAL vs EXTERNAL MARKS CORRELATION", THEME['primary']))

btn_corr = make_btn("📉 Plot", THEME['primary'])
corr_out = make_output()

def plot_corr(b):
    with corr_out:
        clear_output()
        df = query_df("SELECT internal, external, total FROM marks")
        fig, axes = plt.subplots(1, 2, figsize=(12, 5))

        axes[0].scatter(df['internal'], df['total'], alpha=0.6, color='#4A90D9', s=60)
        z = np.polyfit(df['internal'], df['total'], 1)
        x = np.linspace(df['internal'].min(), df['internal'].max(), 100)
        axes[0].plot(x, np.poly1d(z)(x), 'r--', linewidth=1.5)
        axes[0].set_xlabel("Internal (20)"); axes[0].set_ylabel("Total (100)")
        axes[0].set_title("Internal vs Total", fontweight='bold')
        r1 = round(np.corrcoef(df['internal'], df['total'])[0,1], 3)
        axes[0].text(0.05, 0.92, f"r = {r1}", transform=axes[0].transAxes, color='red', fontweight='bold')

        axes[1].scatter(df['external'], df['total'], alpha=0.6, color='#E74C3C', s=60)
        z2 = np.polyfit(df['external'], df['total'], 1)
        x2 = np.linspace(df['external'].min(), df['external'].max(), 100)
        axes[1].plot(x2, np.poly1d(z2)(x2), 'b--', linewidth=1.5)
        axes[1].set_xlabel("External (50)"); axes[1].set_ylabel("Total (100)")
        axes[1].set_title("External vs Total", fontweight='bold')
        r2 = round(np.corrcoef(df['external'], df['total'])[0,1], 3)
        axes[1].text(0.05, 0.92, f"r = {r2}", transform=axes[1].transAxes, color='blue', fontweight='bold')

        plt.suptitle("Marks Correlation Analysis", fontsize=13, fontweight='bold')
        plt.tight_layout(); plt.show()

btn_corr.on_click(plot_corr)
display(btn_corr)
display(corr_out)


Button(description='📉 Plot', layout=Layout(height='36px', width='180px'), style=ButtonStyle(button_color='#4A9…

Output(layout=Layout(border_bottom='1px solid #ddd', border_left='1px solid #ddd', border_right='1px solid #dd…

In [49]:

display(styled_html("🔔 NOTIFICATION CENTER", THEME['danger']))

btn_notify = make_btn("🔔 Check Alerts", THEME['danger'])
notify_out = make_output()

def check_notifications(b):
    with notify_out:
        clear_output()
        alerts = []

        # Low attendance
        df_low = query_df("""
            SELECT s.name, s.email, a.subject,
                   ROUND(100.0*SUM(CASE WHEN a.status='Present' THEN 1 ELSE 0 END)/COUNT(*),1) as pct
            FROM attendance a JOIN students s ON a.student_id=s.student_id
            GROUP BY a.student_id, a.subject HAVING pct < 75
        """)
        for _, row in df_low.iterrows():
            alerts.append(f'🔴 <b>Low Attendance</b>: {row["name"]} — {row["subject"]}: {row["pct"]}%')

        # Failed subjects
        df_fail = query_df("""
            SELECT s.name, m.subject FROM marks m JOIN students s ON m.student_id=s.student_id WHERE m.grade='F'
        """)
        for _, row in df_fail.iterrows():
            alerts.append(f'❌ <b>Failure Alert</b>: {row["name"]} failed {row["subject"]}')

        if not alerts:
            display(HTML('<p style="color:green;font-size:15px">✅ No alerts! All students are on track.</p>'))
        else:
            display(HTML(f'<b>Total Alerts: {len(alerts)}</b>'))
            for alert in alerts[:20]:
                display(HTML(f'<div style="background:#FFF3CD;border-left:4px solid #E74C3C;padding:8px 12px;margin:4px 0;border-radius:4px">{alert}</div>'))

btn_notify.on_click(check_notifications)
display(btn_notify)
display(notify_out)

Button(description='🔔 Check Alerts', layout=Layout(height='36px', width='180px'), style=ButtonStyle(button_col…

Output(layout=Layout(border_bottom='1px solid #ddd', border_left='1px solid #ddd', border_right='1px solid #dd…

In [50]:

display(HTML("""
<div style="background:linear-gradient(135deg,#2C3E50,#4A90D9);color:white;padding:24px;border-radius:12px;font-family:Arial">
  <h2 style="text-align:center;margin:0 0 12px">🎓 Student ERP Management System</h2>
  <p style="text-align:center;margin:0 0 16px;opacity:0.9">Built with ipywidgets · sqlite3 · matplotlib · pandas</p>
  <table style="width:100%;border-collapse:collapse;font-size:14px">
    <tr style="background:rgba(255,255,255,0.15)">
      <th style="padding:10px;text-align:left">Module</th>
      <th style="padding:10px;text-align:left">Cells</th>
      <th style="padding:10px;text-align:left">Features</th>
    </tr>
    <tr style="background:rgba(255,255,255,0.05)">
      <td style="padding:8px 10px">🏗 Database Setup</td>
      <td>1–5</td>
      <td>SQLite schema, seeding, helpers</td>
    </tr>
    <tr style="background:rgba(255,255,255,0.10)">
      <td style="padding:8px 10px">👤 Student CRUD</td>
      <td>6–13</td>
      <td>Register, View, Search, Update, Delete, Profile</td>
    </tr>
    <tr style="background:rgba(255,255,255,0.05)">
      <td style="padding:8px 10px">📅 Attendance</td>
      <td>14–17</td>
      <td>Mark, View, Summary, Bar Chart</td>
    </tr>
    <tr style="background:rgba(255,255,255,0.10)">
      <td style="padding:8px 10px">📝 Marks + GPA</td>
      <td>18–22</td>
      <td>Entry, Scorecard, Radar, GPA, Trend</td>
    </tr>
    <tr style="background:rgba(255,255,255,0.05)">
      <td style="padding:8px 10px">📊 Analytics</td>
      <td>23–29</td>
      <td>Heatmap, Box, Scatter, Dashboard, Leaderboard</td>
    </tr>
    <tr style="background:rgba(255,255,255,0.10)">
      <td style="padding:8px 10px">🔧 Admin Panel</td>
      <td>30–36</td>
      <td>Stats, Low Att, Failures, Promote, Export, Log, Reset</td>
    </tr>
    <tr style="background:rgba(255,255,255,0.05)">
      <td style="padding:8px 10px">🖥 Advanced UI</td>
      <td>37–50</td>
      <td>SQL Terminal, Classifier, Report Card, Tabs, Alerts</td>
    </tr>
  </table>
  <p style="text-align:center;margin-top:16px;opacity:0.8">▶ Run cells in order. All data persists in <code>student_erp.db</code></p>
</div>
"""))

Module,Cells,Features
🏗 Database Setup,1–5,"SQLite schema, seeding, helpers"
👤 Student CRUD,6–13,"Register, View, Search, Update, Delete, Profile"
📅 Attendance,14–17,"Mark, View, Summary, Bar Chart"
📝 Marks + GPA,18–22,"Entry, Scorecard, Radar, GPA, Trend"
📊 Analytics,23–29,"Heatmap, Box, Scatter, Dashboard, Leaderboard"
🔧 Admin Panel,30–36,"Stats, Low Att, Failures, Promote, Export, Log, Reset"
🖥 Advanced UI,37–50,"SQL Terminal, Classifier, Report Card, Tabs, Alerts"
